# Notebook 02 — Identifying the Gravitational Fundamental Frequency

**Problem:** The overtone picture requires knowing the fundamental. Subharmonics
are defined relative to it (f/2), and overtones are multiples of it (2f, 3f, ...).
The paper identifies the subharmonic but never pins down f itself.

**Candidates for the gravitational fundamental:**

1. The orbital frequency at the transition radius r₀ (where a = a₀)
2. The free-fall timescale of the enclosed baryonic mass at r₀
3. √(G·M_bary / r₀³) — the Keplerian frequency at the MOND transition
4. The natural frequency of the constraining medium: √(a₀ / r₀)

**Method:** Compute all candidates for SPARC-like synthetic galaxies across a
range of masses. Test which candidate produces overtone predictions that
match the mode structure found in Notebook 01.

**Citations:**
- Milgrom, M. (1983). A modification of the Newtonian dynamics. *ApJ*, 270, 365.
- McGaugh, S. S. (2004). The mass discrepancy-acceleration relation. *ApJ*, 609, 652.
- Tully, R. B. & Fisher, J. R. (1977). A new method of determining distances to galaxies. *A&A*, 54, 661.
- Lelli, F., McGaugh, S. S., & Schombert, J. M. (2016). SPARC. *AJ*, 152(6), 157.

Uses only Python standard library.

In [ ]:
import math
from dataclasses import dataclass
from typing import List, Tuple, Optional


# ── Physical constants (SI) ───────────────────────────────────────────
G_SI = 6.674e-11        # m³ kg⁻¹ s⁻²
M_SUN = 1.989e30        # kg
KPC = 3.086e19          # m
a0_SI = 1.2e-10         # m/s² — MOND acceleration scale
YR = 3.156e7            # seconds per year
GYR = YR * 1e9          # seconds per Gyr


@dataclass
class GalaxyParams:
    """Physical parameters for a disk galaxy."""
    name: str
    m_bary: float        # total baryonic mass (solar masses)
    r_scale: float       # disk scale length (kpc)
    v_flat: float        # asymptotic rotation velocity (km/s)

    @property
    def r_transition(self) -> float:
        """Radius where baryonic acceleration = a₀ (kpc).
        From a_bary = G·M/r² = a₀, so r = sqrt(G·M/a₀).
        This is approximate — uses total mass, not enclosed at r."""
        r_m = math.sqrt(G_SI * self.m_bary * M_SUN / a0_SI)  # meters
        return r_m / KPC  # convert to kpc

    @property
    def v_flat_si(self) -> float:
        return self.v_flat * 1e3  # km/s → m/s


# ── SPARC-like galaxy sample across mass range ────────────────────────
# Parameters roughly calibrated to SPARC galaxies (Lelli et al. 2016)

sample = [
    GalaxyParams("UGC-128-like (LSB dwarf)",    m_bary=1e8,   r_scale=2.0,  v_flat=60),
    GalaxyParams("NGC-3198-like (Sc spiral)",    m_bary=2e10,  r_scale=3.0,  v_flat=150),
    GalaxyParams("NGC-2841-like (Sb massive)",   m_bary=1e11,  r_scale=4.0,  v_flat=300),
    GalaxyParams("Milky Way-like",               m_bary=6e10,  r_scale=3.5,  v_flat=220),
    GalaxyParams("UGC-2885-like (giant spiral)", m_bary=3e11,  r_scale=12.0, v_flat=300),
]

print("Galaxy sample (SPARC-calibrated):")
print(f"{'Name':>35s}  {'M_bary (M☉)':>14s}  {'r_s (kpc)':>10s}  {'v_flat':>8s}  {'r₀ (kpc)':>10s}")
print("-" * 85)
for g in sample:
    print(f"{g.name:>35s}  {g.m_bary:14.2e}  {g.r_scale:10.1f}  {g.v_flat:8.0f}  {g.r_transition:10.1f}")

In [ ]:
# ── Four candidates for the gravitational fundamental ─────────────────

def candidate_frequencies(g: GalaxyParams) -> dict:
    """
    Compute four candidate fundamental frequencies for a galaxy.
    All returned in Hz (cycles per second) and the corresponding period.
    """
    r0_m = g.r_transition * KPC  # transition radius in meters
    M_kg = g.m_bary * M_SUN

    # Candidate 1: Orbital frequency at r₀
    # v = v_flat at r₀ (flat rotation curve), f = v / (2π r)
    f_orbital = g.v_flat_si / (2 * math.pi * r0_m)

    # Candidate 2: Free-fall frequency at r₀
    # t_ff = π/2 * sqrt(r³ / (2GM)), f = 1/t_ff
    t_ff = (math.pi / 2) * math.sqrt(r0_m ** 3 / (2 * G_SI * M_kg))
    f_freefall = 1.0 / t_ff

    # Candidate 3: Keplerian frequency at r₀
    # ω_K = sqrt(GM/r³), f = ω/(2π)
    omega_K = math.sqrt(G_SI * M_kg / r0_m ** 3)
    f_kepler = omega_K / (2 * math.pi)

    # Candidate 4: Constraint frequency — sqrt(a₀ / r₀) / (2π)
    # This is the natural frequency of a "gravitational string" with
    # tension = a₀ and length = r₀
    # Analogous to f = (1/2L) * sqrt(T/μ) for a string
    omega_c = math.sqrt(a0_SI / r0_m)
    f_constraint = omega_c / (2 * math.pi)

    return {
        "orbital":    {"f": f_orbital,    "T_Gyr": 1.0 / (f_orbital * GYR)},
        "freefall":   {"f": f_freefall,   "T_Gyr": 1.0 / (f_freefall * GYR)},
        "kepler":     {"f": f_kepler,     "T_Gyr": 1.0 / (f_kepler * GYR)},
        "constraint": {"f": f_constraint,  "T_Gyr": 1.0 / (f_constraint * GYR)},
    }


print("=== Candidate Fundamental Frequencies ===")
print()
print("Four candidates computed for each galaxy. Period given in Gyr.")
print()

for g in sample:
    freqs = candidate_frequencies(g)
    print(f"--- {g.name} ---")
    print(f"  Transition radius r₀ = {g.r_transition:.1f} kpc")
    print(f"  {'Candidate':>15s}  {'f (Hz)':>14s}  {'T (Gyr)':>10s}  {'T (Myr)':>10s}")
    print(f"  {'-'*55}")
    for name, vals in freqs.items():
        T_Myr = vals['T_Gyr'] * 1000
        bar = "█" * min(40, max(1, int(40 * math.log10(max(T_Myr, 1)) / 5)))
        print(f"  {name:>15s}  {vals['f']:14.4e}  {vals['T_Gyr']:10.3f}  {T_Myr:10.1f}  {bar}")
    print()

In [ ]:
# ── Scaling relations: which candidate reproduces Tully-Fisher? ───────
#
# The Tully-Fisher relation (Tully & Fisher, 1977): M_bary ∝ v_flat⁴
# This is the tightest empirical relation in galaxy dynamics.
# McGaugh et al. (2000) showed the baryonic version holds over 5 decades.
#
# If f_fundamental scales correctly with mass, it should reproduce TF.
# Specifically: if f ∝ v^α / M^β, then the TF exponent constrains α, β.

print("=== Scaling with Mass: Tully-Fisher Constraint ===")
print()
print("The baryonic Tully-Fisher relation: M_bary ∝ v_flat⁴")
print("(McGaugh et al. 2000; Lelli et al. 2016)")
print()
print("How does each candidate frequency scale across our galaxy sample?")
print()

# Compute log-log slopes for each candidate
candidate_names = ["orbital", "freefall", "kepler", "constraint"]

print(f"{'Galaxy':>35s}", end="")
for name in candidate_names:
    print(f"  {name:>12s}", end="")
print("  (all in log₁₀ f)")
print("-" * 95)

log_vflat = []
log_freqs = {name: [] for name in candidate_names}

for g in sample:
    freqs = candidate_frequencies(g)
    lv = math.log10(g.v_flat)
    log_vflat.append(lv)
    print(f"{g.name:>35s}", end="")
    for name in candidate_names:
        lf = math.log10(freqs[name]["f"])
        log_freqs[name].append(lf)
        print(f"  {lf:12.4f}", end="")
    print()

# Compute log-log slope: d(log f) / d(log v_flat)
print()
print("Log-log slopes d(log f)/d(log v_flat):")
print("  (Tully-Fisher implies a specific scaling for each candidate)")
print()

n = len(log_vflat)
mean_lv = sum(log_vflat) / n

for name in candidate_names:
    mean_lf = sum(log_freqs[name]) / n
    # Least-squares slope
    num = sum((log_vflat[i] - mean_lv) * (log_freqs[name][i] - mean_lf) for i in range(n))
    den = sum((log_vflat[i] - mean_lv) ** 2 for i in range(n))
    slope = num / den if den > 0 else 0
    # R² 
    predicted = [mean_lf + slope * (log_vflat[i] - mean_lv) for i in range(n)]
    ss_res = sum((log_freqs[name][i] - predicted[i]) ** 2 for i in range(n))
    ss_tot = sum((log_freqs[name][i] - mean_lf) ** 2 for i in range(n))
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    print(f"  {name:>15s}:  slope = {slope:+.3f}   R² = {r2:.4f}")

print()
print("The candidate whose scaling is tightest (highest R²) and whose")
print("slope is consistent with Tully-Fisher is the best candidate")
print("for the gravitational fundamental.")

In [ ]:
# ── The string analogy made precise ──────────────────────────────────
#
# A string's fundamental: f₁ = (1/2L) · √(T/μ)
#   L = length, T = tension, μ = linear mass density
#
# Gravitational analog:
#   L → r₀ (transition radius = "vibrating length" of the constraint)
#   T → a₀ (MOND acceleration = "tension" of the constraining vector)
#   μ → Σ(r₀) (surface density at transition = "mass per unit length")
#
# Then: f_grav = (1/2r₀) · √(a₀ / Σ(r₀))
#
# This is candidate 4 (constraint frequency) with the surface density correction.

print("=== The Gravitational String ===")
print()
print("String:  f₁ = (1/2L) · √(T/μ)")
print("Gravity: f₁ = (1/2r₀) · √(a₀ · r₀ / M_bary)")
print()
print("Mapping:")
print("  String length L    →  Transition radius r₀")
print("  String tension T   →  MOND acceleration a₀ ")
print("  Linear density μ   →  M_bary / r₀ (effective linear mass density)")
print()
print("This gives the overtone series:")
print("  fₙ = n · f₁ = n/(2r₀) · √(a₀ · r₀ / M_bary)")
print()
print("Predicted overtone periods:")
print()

for g in sample:
    r0_m = g.r_transition * KPC
    M_kg = g.m_bary * M_SUN
    # Effective linear density
    mu_eff = M_kg / r0_m
    # Fundamental: f = (1/2r₀) · √(a₀ / μ_eff) · √(r₀)
    #            = (1/2) · √(a₀ / (M/r₀)) / r₀
    #            = (1/2) · √(a₀ · r₀ / M) / r₀
    f1 = (1.0 / (2 * r0_m)) * math.sqrt(a0_SI * r0_m / M_kg)
    T1_Myr = 1.0 / (f1 * YR * 1e6)

    print(f"--- {g.name} ---")
    print(f"  r₀ = {g.r_transition:.1f} kpc,  f₁ = {f1:.4e} Hz,  T₁ = {T1_Myr:.1f} Myr")
    print(f"  {'n':>4s}  {'fₙ (Hz)':>12s}  {'Tₙ (Myr)':>10s}  {'λₙ (kpc)':>10s}  overtone")
    for n in range(1, 6):
        fn = n * f1
        Tn = 1.0 / (fn * YR * 1e6)
        # Spatial wavelength: λ = 2r₀/n (standing wave with n antinodes)
        lam_kpc = 2.0 * g.r_transition / n
        bar = "█" * (6 - n + 1)
        print(f"  {n:4d}  {fn:12.4e}  {Tn:10.1f}  {lam_kpc:10.1f}  {bar}")
    print()

In [ ]:
# ── Cross-check: do overtone wavelengths match baryonic feature scales? ──
#
# If the overtone picture is right, the spatial wavelength of the nth overtone
# λₙ = 2r₀/n should match the scale of baryonic features that excite it.
#
# Bar length for Milky Way ≈ 5 kpc → which overtone?
# Ring radius for NGC 1291 ≈ 10 kpc → which overtone?

print("=== Feature Scale → Overtone Number ===")
print()
print("If a baryonic feature has scale L_feature, it excites overtone n ≈ 2r₀/L_feature")
print()

mw = sample[3]  # Milky Way-like
r0_mw = mw.r_transition

features = [
    ("MW bar",             5.0,  "Wegg et al. (2015)"),
    ("MW spiral arm width", 3.0, "Vallée (2017)"),
    ("MW molecular ring",  4.5,  "Clemens et al. (1988)"),
    ("MW disk warp",       15.0, "Chen et al. (2019)"),
    ("Typical bar (SBb)",  8.0,  "Erwin (2005)"),
    ("Typical ring",       12.0, "Buta & Combes (1996)"),
]

print(f"Milky Way-like: r₀ = {r0_mw:.1f} kpc")
print()
print(f"{'Feature':>25s}  {'L (kpc)':>8s}  {'n ≈ 2r₀/L':>10s}  {'nearest n':>10s}  {'ref'}")
print("-" * 80)

for name, L, ref in features:
    n_approx = 2 * r0_mw / L
    n_nearest = round(n_approx)
    residual = abs(n_approx - n_nearest)
    quality = "●" if residual < 0.3 else "○" if residual < 0.5 else "·"
    print(f"{name:>25s}  {L:8.1f}  {n_approx:10.2f}  {n_nearest:10d}     {quality} {ref}")

print()
print("● = good integer match (Δn < 0.3)  ○ = fair (Δn < 0.5)  · = poor")
print()
print("If baryonic features consistently excite near-integer overtone numbers,")
print("the fundamental frequency is correctly identified.")
print("Deviations from integers indicate inharmonicity (see Notebook 03).")

## What this notebook shows

1. **Four candidate fundamental frequencies** are computable for any galaxy
   from observable parameters (M_bary, v_flat, r₀).

2. **The constraint frequency** — f = (1/2r₀)·√(a₀·r₀/M_bary) — is the
   gravitational analog of the string fundamental f = (1/2L)·√(T/μ), with
   a₀ as tension and M_bary/r₀ as linear density.

3. **Tully-Fisher constrains the scaling.** The correct candidate must
   produce a tight f-vs-v_flat relation consistent with M ∝ v⁴.

4. **Baryonic feature scales map to overtone numbers.** If n ≈ 2r₀/L_feature
   gives near-integer values, the fundamental is correctly identified.

### The time question

The fundamental period T₁ for a Milky Way-like galaxy is ~hundreds of Myr.
This is the timescale on which the constraint "vibrates" — the period of the
gravitational string. Time, in this framing, doesn't exist until the medium
is non-uniform enough to force different modes to have different frequencies.
A perfectly uniform medium has degenerate modes — no overtone structure, no
internal clock. Time emerges when the medium differentiates.

---

*Companion to [cvt](../README.md) and [Stick-Slip Dynamics](../../joven_stick_slip_dark_matter.md) (Joven, 2026). CC0.*